# LIMPIEZA  DE DATOS

### IMPORTAR LIBRERIAS

In [0]:
# !pip install duckdb


In [0]:
import datetime
import duckdb
import pandas as pd
horainicio = datetime.datetime.now()

### RUTAS PRINCIPALES

In [0]:
# RUTA RAIZ PRINCIPAL DEL PROYECTO EN DATABRICKS CLOUD
PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"

PATH_PROJECT = f"{PATH_DATABRICKS}/"

In [0]:
## VERIFICAR LA EXISTENCIA DE LA RUTA COMPARTIDA  AÑADIDA DATA-BRICKS CLOUD
archivos_y_carpetas = dbutils.fs.ls(PATH_DATABRICKS)
# Para imprimir cada elemento:
for elemento in archivos_y_carpetas:
    print(elemento.path)

### LIMPIEZA Y OBTENCION CONSOLIDADO DE DATOS  TBL GUI

In [0]:
### ENSAMBLE DE RUTAS PARA LOS ARCHIVOS PARQUET
RUTA_BATCH_LOG_GUIAS = f"{PATH_PROJECT}LOG_GUIAS/LOG_GUIAS*.parquet"
RUTA_FINAL_LOG_GUIAS = f"{PATH_PROJECT}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"

In [0]:
# TRANSPORTE Y SEGMENTACION DE GUIAS DE BOGOTA (ID_CIUDAD_DESTINO = 10) Y CON NUMERO DE GUIA >= 8 Y ESTADO DE ENVIO >= 3
con = duckdb.connect()

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{RUTA_BATCH_LOG_GUIAS}')
        WHERE ID_CIUDAD_DESTINO = 10
        AND LEN(NUMERO_GUIA::VARCHAR) >= 8
        AND ID_ESTADOENVIO >= 3
    )
    TO '{RUTA_FINAL_LOG_GUIAS}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Parquet consolidado correctamente.")

con.close()

In [0]:
df_resultado = duckdb.sql(f"""
    SELECT COUNT(1) AS TOTAL_REGISTROS
    FROM read_parquet('{ RUTA_FINAL_LOG_GUIAS }')
""").df()
display(df_resultado)

### LIMPIEZA Y OBTENCION CONSOLIDADO DE DATOS  TBL MOVIMIENTOS

In [0]:
### ENSAMBLE DE RUTAS PARA LOS ARCHIVOS PARQUET
RUTA_BATCH_LOG_SEGUIMIENTO_ENVIO = f"{PATH_PROJECT}LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO*.parquet"
RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO = f"{PATH_PROJECT}LOG_SEGUIMIENTO_ENVIO/RAW/LOG_SEGUIMIENTO_ENVIO_BOGOTA.parquet"
RUTA_RELACION = f"{PATH_PROJECT}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"

In [0]:
# pRUEBAS SCRIPT MOVIMIENTO SEGUIMIENTO DE GUIAS
df_resultado = duckdb.sql(f"""
        with CTE AS(
        SELECT
        	ROW_NUMBER ( )
				OVER (
					PARTITION BY G.NUMERO_GUIA
					order by S.FECHA_HORA_MOVIMIENTO desc
					)  AS ROW_ID_SE --> ULTIMO MOVIMIENTO
        ,S.*
        FROM read_parquet('{RUTA_RELACION}') G
        INNER JOIN read_parquet('{RUTA_BATCH_LOG_SEGUIMIENTO_ENVIO}') S
        ON G.NUMERO_GUIA = S.NUMERO_GUIA
        )
        SELECT
            COUNT(1)
        FROM CTE WHERE ROW_ID_SE = 1
           --LIMIT 5
""").df()
display(df_resultado)

In [0]:
# TRANSPORTE Y SEGMENTACION DE LOS MOVIMIENTOS DE GUIAS DE BOGOTA
con = duckdb.connect()

con.execute(f"""
    COPY (
        with CTE AS(
        SELECT
        	ROW_NUMBER ( )
				OVER (
					PARTITION BY G.NUMERO_GUIA
					order by S.FECHA_HORA_MOVIMIENTO desc
					)  AS ROW_ID_SE --> ULTIMO MOVIMIENTO
        ,S.*
        FROM read_parquet('{RUTA_RELACION}') G
        INNER JOIN read_parquet('{RUTA_BATCH_LOG_SEGUIMIENTO_ENVIO}') S
        ON G.NUMERO_GUIA = S.NUMERO_GUIA
        )SELECT * FROM CTE WHERE ROW_ID_SE = 1
    )
    TO '{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Parquet consolidado correctamente.")

con.close()

In [0]:
# PRUEBAS SCRIPT VALIDAR NUMERO DE GUIAS POR MOVIMIENTO SEGUIMIENTO DE GUIAS
df_resultado = duckdb.sql(f"""
        SELECT
            COUNT(1)
        FROM read_parquet('{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}')
""").df()
display(df_resultado)

In [0]:
### ENSAMBLE DE RUTAS PARA LOS ARCHIVOS PARQUET
RUTA_BATCH_LOG_DOCUMENTOS = f"{PATH_PROJECT}LOG_DOCUMENTOS/LOG_DOCUMENTOS*.parquet"
RUTA_FINAL_LOG_DOCUMENTOS = f"{PATH_PROJECT}LOG_DOCUMENTOS/RAW/LOG_DOCUMENTOS_BOGOTA.parquet"
RUTA_RELACION_SE = f"{PATH_PROJECT}LOG_SEGUIMIENTO_ENVIO/RAW/LOG_SEGUIMIENTO_ENVIO_BOGOTA.parquet"

In [0]:
# TRANSPORTE Y SEGMENTACION DE LOS MANIFIESTOS DE BOGOTA
con = duckdb.connect()

con.execute(f"""
    COPY (
        with CTE AS(
        SELECT
        	ROW_NUMBER ( )
				OVER (
					PARTITION BY
                                SEG.ID_PROCESO,
                                SEG.NUMERO_MOVIMI,
                                SEG.CONSECUT_MOVIMI
					order by DOC.FECHA_HORA_MOVIMI desc
					)  AS ROW_ID_SE --> ULTIMO MOVIMIENTO
        ,DOC.*
        FROM read_parquet('{RUTA_RELACION_SE}') AS SEG
        INNER JOIN read_parquet('{RUTA_BATCH_LOG_DOCUMENTOS}') AS DOC
	         ON SEG.ID_PROCESO      =   DOC.ID_PROCESO
            AND SEG.NUMERO_MOVIMI   =   DOC.NUMERO_MOVIMI
            AND SEG.CONSECUT_MOVIMI =   DOC.CONSECUT_MOVIMI
        )
        SELECT
            *
            FROM CTE WHERE ROW_ID_SE = 1
    )
    TO '{RUTA_FINAL_LOG_DOCUMENTOS}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Parquet consolidado correctamente.")

con.close()

In [0]:
# PRUEBAS SCRIPT VALIDAR NUMERO DE GUIAS POR MOVIMIENTO SEGUIMIENTO DE GUIAS
df_resultado = duckdb.sql(f"""
        SELECT
            COUNT(1)
        FROM read_parquet('{RUTA_FINAL_LOG_DOCUMENTOS}')
""").df()
display(df_resultado)

# TRANSFORMACION E INTEGRACION DE DATOS

In [0]:
df_GUIAS = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_FINAL_LOG_GUIAS}'
    LIMIT 5
    """).df()
display(df_GUIAS)

In [0]:
print(df_GUIAS.columns)

In [0]:
df_MOV = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}'
    LIMIT 5
    """).df()
display(df_MOV)

In [0]:
print(df_MOV.columns)

In [0]:
df_DOC = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_FINAL_LOG_DOCUMENTOS}'
    LIMIT 5
    """).df()
display(df_DOC)

In [0]:
print(df_DOC.columns)

## Transformacion contruccion Tabla Base de trabajo  

In [0]:
# CONFIGURAR RUTA FINAL BASE CONSOLIDADA DE LOS 3 PARQUETS
RUTA_FINAL_TABLA_BASE = f"/Volumes/proyectogrado/source/sourcedb/LOG_CONSOLIDADO/LOG_BASE_BOGOTA.parquet"
print(RUTA_FINAL_TABLA_BASE)

In [0]:
# PRUEBAS SCRIPT VALORES DUPLICADOS
'''
RUTA_FINAL_LOG_GUIAS
RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO
RUTA_FINAL_LOG_DOCUMENTOS
'''

df_resultado = duckdb.sql(f"""
        SELECT
            NUMERO_GUIA, COUNT(1) AS TOTAL_REGISTROS
        FROM '{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}'
        GROUP BY NUMERO_GUIA
        HAVING COUNT(1) > 1
""").df()
if df_resultado.empty:
    print("No se encontraron registros duplicados.")
else:
    print(df_resultado)
#--------------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------------
df_resultado = duckdb.sql(f"""
        SELECT
            ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI, COUNT(1) AS TOTAL_REGISTROS
        FROM '{RUTA_FINAL_LOG_DOCUMENTOS}'
        GROUP BY ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI
        HAVING COUNT(1) > 1
""").df()
if df_resultado.empty:
    print("No se encontraron registros duplicados.")
else:
    print(df_resultado)
#--------------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------------
df_resultado = duckdb.sql(f"""
        SELECT SEG.NUMERO_GUIA, COUNT(1) AS TOTAL_REGISTROS
                FROM read_parquet('{RUTA_FINAL_LOG_GUIAS}') AS GUI
                INNER JOIN read_parquet('{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}') AS SEG
                    ON SEG.NUMERO_GUIA = GUI.NUMERO_GUIA
                GROUP BY SEG.NUMERO_GUIA
                HAVING COUNT(1) > 1
                    """).df()
if df_resultado.empty:
    print("No se encontraron registros duplicados.")
else:
    print(df_resultado)
#--------------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------------------------
df_resultado = duckdb.sql(f"""
        SELECT
                 DOC.ID_PROCESO
                ,DOC.NUMERO_MOVIMI
                ,DOC.CONSECUT_MOVIMI
                ,COUNT(1) AS TOTAL_REGISTROS
                FROM read_parquet('{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}') AS SEG
                    INNER JOIN read_parquet('{RUTA_FINAL_LOG_DOCUMENTOS}') AS DOC
                        ON DOC.ID_PROCESO   = SEG.ID_PROCESO
                    AND DOC.NUMERO_MOVIMI   = SEG.NUMERO_MOVIMI
                    AND DOC.CONSECUT_MOVIMI = SEG.CONSECUT_MOVIMI
                GROUP BY DOC.ID_PROCESO, DOC.NUMERO_MOVIMI, DOC.CONSECUT_MOVIMI
                HAVING COUNT(1) > 1
                    """).df()
# En lugar de display(df_resultado), usa esto:
if df_resultado.empty:
    print("No se encontraron registros duplicados.")
else:
    print(df_resultado)

In [0]:
# TRANSPORTE Y SEGMENTACION DE LOS MANIFIESTOS DE BOGOTA
con = duckdb.connect()
con.execute(f"""
    COPY (
                SELECT DISTINCT
                        ---------------------------------->Tikets Guia,
                        ---GUI.NUMERO_GUIA AS G_NUMERO_GUIA,
                        GUI.ID_TIPOGUIA AS G_ID_TIPOGUIA,
                        GUI.ID_PAIS_ORIGEN AS G_ID_PAIS_ORIGEN,
                        GUI.ID_CIUDAD_ORIGEN  AS G_ID_CIUDAD_ORIGEN ,
                        GUI.ID_PAIS_DESTINO  AS G_ID_PAIS_DESTINO ,
                        GUI.ID_CIUDAD_DESTINO AS G_ID_CIUDAD_DESTINO,
                        GUI.ID_ESTADOENVIO  AS G_ID_ESTADOENVIO ,
                        GUI.ID_ESTADOGUIA AS G_ID_ESTADOGUIA,
                        GUI.ID_TIEMPO  AS G_ID_TIEMPO ,
                        GUI.ID_MEDTRANS  AS G_ID_MEDTRANS ,
                        GUI.ID_PRODUCTO  AS G_ID_PRODUCTO ,
                        GUI.ID_SUBPRODUCTO AS G_ID_SUBPRODUCTO,
                        GUI.DIRECCION_ESTANDAR AS G_DIRECCION_ESTANDAR,
                        GUI.DIRECCION_REMITE_GUIA AS G_DIRECCION_REMITE_GUIA,
                        GUI.DIRECCION_DESTINA_GUIA AS G_DIRECCION_DESTINA_GUIA,
                        ---------------------------------->MOVS,
                        SEG.ID_PROCESO  AS SEG_ID_PROCESO ,
                        SEG.NUMERO_MOVIMI  AS SEG_NUMERO_MOVIMI ,
                        SEG.CONSECUT_MOVIMI AS SEG_CONSECUT_MOVIMI,
                        SEG.FECHA_HORA_MOVIMIENTO AS SEG_FECHA_HORA_MOVIMIENTO,
                        SEG.ID_CONCEPTO AS SEG_ID_CONCEPTO,
                        SEG.ID_TIPOENVIO AS SEG_ID_TIPOENVIO,
                        SEG.PESO_ENVIO AS SEG_PESO_ENVIO,
                        SEG.LARGO_ENVIO  AS SEG_LARGO_ENVIO ,
                        SEG.ANCHO_ENVIO  AS SEG_ANCHO_ENVIO ,
                        SEG.ALTO_ENVIO AS SEG_ALTO_ENVIO,
                        SEG.NUMERO_PIEZAS AS SEG_NUMERO_PIEZAS,
                        ---------------------------------->DOCS,
                        DOC.ID_TIPO_ZONA_URBA  AS DOC_ID_TIPO_ZONA_URBA ,
                        DOC.ID_PAIS_ZONA_URBA  AS DOC_ID_PAIS_ZONA_URBA ,
                        DOC.ID_CIUDAD_ZONA_URBA AS DOC_ID_CIUDAD_ZONA_URBA,
                        DOC.ID_ZONA_URBA  AS DOC_ID_ZONA_URBA ,
                        DOC.ID_PAIS_DIVISION_ORIGEN  AS DOC_ID_PAIS_DIVISION_ORIGEN ,
                        DOC.ID_CIUDAD_DIVISION_ORIGEN AS DOC_ID_CIUDAD_DIVISION_ORIGEN,
                        DOC.ID_DIVISION_ORIGEN  AS DOC_ID_DIVISION_ORIGEN ,
                        DOC.ID_PAIS_DESTINO  AS DOC_ID_PAIS_DESTINO ,
                        DOC.ID_CIUDAD_DESTINO AS DOC_ID_CIUDAD_DESTINO,
                        DOC.ID_DIVISION_DESTINO AS DOC_ID_DIVISION_DESTINO
                FROM read_parquet('{RUTA_FINAL_LOG_GUIAS}') AS GUI
                INNER JOIN read_parquet('{RUTA_FINAL_LOG_SEGUIMIENTO_ENVIO}') AS SEG
                        ON SEG.NUMERO_GUIA = GUI.NUMERO_GUIA
                INNER JOIN read_parquet('{RUTA_FINAL_LOG_DOCUMENTOS}') AS DOC
                        ON DOC.ID_PROCESO = SEG.ID_PROCESO
                    AND DOC.NUMERO_MOVIMI = SEG.NUMERO_MOVIMI
                    AND DOC.CONSECUT_MOVIMI = SEG.CONSECUT_MOVIMI
    )
    TO '{RUTA_FINAL_TABLA_BASE}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Parquet consolidado correctamente.")

con.close()

In [0]:
# PRUEBAS SCRIPT VALIDAR NUMERO DE REGISTROS EN LA TABLA BASE CONSOLIDADA
df_resultado = duckdb.sql(f"""
        SELECT
         COUNT(1) AS TOTAL_REGISTROS
           FROM '{RUTA_FINAL_TABLA_BASE}'
""").df()
display(df_resultado)

df_resultado = duckdb.sql(f"""
        SELECT *
           FROM '{RUTA_FINAL_TABLA_BASE}'
           LIMIT 5
""").df()
display(df_resultado)
print(df_resultado.columns)

In [0]:
horafin = datetime.datetime.now()
Duracion = horafin - horainicio
print(f"Tiempo de ejecución: {Duracion}")